### PTE Prefetcher
* Translation-Only Tree-Aware Micro-Prefetcher
  - Goal: reduce PTW stall time, not maximize raw TXVC hit rate.(?)
  - Scope: only run when type == TRANSLATION.
  - Aggressiveness: start with degree 1, confidence-gated.
  - Safety: strict throttles and usefulness feedback.

* Core predictor components:

  * Translation stream table
    - Key: hashed requester IP plus translation level (if available).
    - State: last line, last delta, 2-bit confidence.
    - Trigger: issue one prefetch when the same delta repeats.

  * Page-table sibling predictor
    - Key: parent page-table page tag.
    - Idea: nearby virtual pages tend to touch nearby child PTEs; prefetch neighboring cache line within same page-table page, not blind global plus-one.
    - Constraint: never cross page-table-page boundary.
  
  * Usefulness controller
    - Track prefetched lines that become demand hits.
    - If rolling accuracy drops below threshold (for example 20%), auto-reduce degree to 0 until recovery.
    - Also gate on MSHR pressure like existing prefetchers do (same style as ip_stride.cc:67).

* Why this is better than naive plus-one:
  - It is still sequential-friendly when locality exists.
  - It avoids always assuming contiguous physical PTE layout.
  - It naturally backs off in long-tail regions.

* Implementation notes:
  - using prefetcher metadata to add is_instr, translation_level 

* TXVC admission filter:
  - Keep 2+ admission for demand fills.
  - For prefetch fills, apply stricter admission:
    - admit only if predictor confidence high, or
    - admit only for upper-level PTEs, or
    - admit into a tiny prefetch probation region (1 way per set).
    - Prevents low-value prefetched PTEs from displacing proven LFU residents.

* Evaluation:
  - Translation MPKI and average PTW latency.
  - IPC
  - Prefetch accuracy, coverage, and timeliness.
  - Pollution indicators: demand miss increase in L2/TXVC after enabling prefetch.
  - Breakdown by PTE level and by instruction/data streams.

  - Success criterion I would use:
    - Keep prefetch accuracy above about 25% and MSHR pressure stable.
    - Accept lower TXVC hit-rate gains if PTW latency and IPC improve; this workload can benefit from latency hiding more than reuse capture.





In [ ]:
s

### Improvements

* Dynamic admission threshold
  - Switch between access threshold 2 and 3 based on recent miss burstiness or queue pressure.

### Morrigan-like
Short answer: partially relevant, but probably too heavy as-is for your use case.

* Learn richer access correlations than simple stride.
  - PTE streams are often non-linear, so richer correlation than plus-one/stride can find extra hits.
  - That can help with long-tail reuse.
  - Use context like walker IP, translation level, and recent deltas to predict next PTE lines.

* Use a tiny correlation table keyed by (IP hash, translation level, page-table-page tag).
  - Degree 1 by default, degree 2 only at high confidence.
  - Add strict usefulness feedback (auto-throttle when accuracy drops).

### Synergy with replacement:
* Separate LFU counters for demand and prefetched lines
* Prefetch hits should need to prove usefulness before getting full LFU weight.